# 📘 Notebook – Tabela 5
# Acesso aos serviços preventivos de saúde das mulheres,
# por decis de renda domiciliar per capita (%)
# PNS 2013 e 2019 — Mulheres com 25 anos ou mais

In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from IPython.display import display

# -------------------------------------------------
# Setup do ambiente
# -------------------------------------------------
sys.path.append(str(Path("..").resolve()))
from service.pns_service import get_dataframe

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [3]:
# -------------------------------------------------
# Carregamento dos dados (gold)
# -------------------------------------------------
variables = [
    "renda_domiciliar_pc",
    "preventivo_fez",
    "mamografia_fez",
]

sources = ["2013", "2019"]

df = get_dataframe(
    variables=variables,
    sources=sources,
    filters={
        "sexo": "2",
        "idade": {"operador": ">=", "valor": 25}
    }
)


In [4]:
# -------------------------------------------------
# Limpeza mínima e garantia de tipos
# -------------------------------------------------

df["renda_domiciliar_pc"] = pd.to_numeric(
    df["renda_domiciliar_pc"], errors="coerce"
)


for col in ["preventivo_fez", "mamografia_fez"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# Remover observações sem renda
df = df.dropna(subset=["renda_domiciliar_pc"])


In [5]:
# -------------------------------------------------
# Construção dos decis de renda (por ano)
# -------------------------------------------------
labels_decis = [
    "1º Decil de renda",
    "2º Decil de renda",
    "3º Decil de renda",
    "4º Decil de renda",
    "5º Decil de renda",
    "6º Decil de renda",
    "7º Decil de renda",
    "8º Decil de renda",
    "9º Decil de renda",
    "10º Decil de renda",
]

# Usar pd.qcut por grupo de ano

df["decil_renda"] = (
    df.groupby("origem")["renda_domiciliar_pc"]
    .transform(
        lambda x: pd.qcut(
            x,
            q=10,
            labels=labels_decis,
            duplicates="drop"
        )
    )
)


In [6]:
# -------------------------------------------------
# Função auxiliar para cálculo percentual
# -------------------------------------------------
def tabela_decil(df, col):
    fez = (df[col].mean() * 100).round(2)
    nao_fez = (100 - fez).round(2)

    idx = pd.Index(
        ["Sim", "Não", "Total"],
        name="Realizou exame"
    )

    return pd.Series(
        [fez, nao_fez, 100.0],
        index=idx
    )


In [7]:
# -------------------------------------------------
# Construção da Tabela 5 (por ano)
# -------------------------------------------------
tabelas = {}

for ano in sorted(df["origem"].unique()):
    df_ano = df[df["origem"] == ano]

    tabela = pd.concat(
        {
            "Papanicolau": (
                df_ano
                .groupby("decil_renda")
                .apply(lambda x: tabela_decil(x, "preventivo_fez"))
            ),
            "Mamografia": (
                df_ano
                .groupby("decil_renda")
                .apply(lambda x: tabela_decil(x, "mamografia_fez"))
            ),
        },
        axis=1
    )

    tabelas[ano] = tabela


In [8]:
# -------------------------------------------------
# Exibição final (formato artigo)
# -------------------------------------------------
for ano, tabela in tabelas.items():
    print(
        f"\nTabela 5 – Acesso aos serviços preventivos de saúde das mulheres "
        f"por decis de renda per capita familiar (%), PNS {ano}"
    )
    display(tabela)



Tabela 5 – Acesso aos serviços preventivos de saúde das mulheres por decis de renda per capita familiar (%), PNS 2013


Papanicolau               Mamografia              
Realizou exame             Sim    Não  Total        Sim    Não  Total
decil_renda                                                          
1º Decil de renda        43.49  56.51  100.0      12.14  87.86  100.0
2º Decil de renda        41.83  58.17  100.0      11.12  88.88  100.0
3º Decil de renda        40.81  59.19  100.0      14.40  85.60  100.0
4º Decil de renda        37.13  62.87  100.0      14.25  85.75  100.0
5º Decil de renda        42.23  57.77  100.0      20.52  79.48  100.0
6º Decil de renda        38.49  61.51  100.0      18.11  81.89  100.0
7º Decil de renda        40.09  59.91  100.0      21.40  78.60  100.0
8º Decil de renda        41.00  59.00  100.0      24.20  75.80  100.0
9º Decil de renda        41.96  58.04  100.0      25.70  74.30  100.0
10º Decil de renda       45.84  54.16  100.0      32.73  67.27  100.0


Tabela 5 – Acesso aos serviços preventivos de saúde das mulheres por decis de renda per capita familiar (%), PNS 2019


Papanicolau               Mamografia              
Realizou exame             Sim    Não  Total        Sim    Não  Total
decil_renda                                                          
1º Decil de renda        42.43  57.57  100.0      12.65  87.35  100.0
2º Decil de renda        38.28  61.72  100.0      12.05  87.95  100.0
3º Decil de renda        39.37  60.63  100.0      16.86  83.14  100.0
4º Decil de renda        36.78  63.22  100.0      16.17  83.83  100.0
5º Decil de renda        36.33  63.67  100.0      18.14  81.86  100.0
6º Decil de renda        48.01  51.99  100.0      28.97  71.03  100.0
7º Decil de renda        39.34  60.66  100.0      23.60  76.40  100.0
8º Decil de renda        41.03  58.97  100.0      26.23  73.77  100.0
9º Decil de renda        45.65  54.35  100.0      31.51  68.49  100.0
10º Decil de renda       46.92  53.08  100.0      35.49  64.51  100.0

In [14]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

def exportar_tabela5_final_perfeita(tabelas, filename="Tabela_05_Revista_Final.pdf"):
    plt.rcParams['font.family'] = 'serif'
    
    with PdfPages(filename) as pdf:
        for ano in sorted(tabelas.keys()):
            fig, ax = plt.subplots(figsize=(12, 10))
            ax.axis('off')

            # 1. TÍTULO (Bem espaçado no topo)
            titulo = f"Tabela 5 - Acesso aos serviços preventivos de saúde das mulheres, por decis de renda per\ncapita familiar (PNS {ano}, em %)"
            plt.text(0.5, 0.95, titulo, ha='center', va='top', fontsize=12, fontweight='bold', transform=ax.transAxes)

            # 2. DADOS
            df = tabelas[ano].reset_index()
            sub_header = ["Decil", "Sim", "Não", "Total", "Sim", "Não", "Total"]
            data = [sub_header] + df.values.tolist()

            # 3. CRIAÇÃO DA TABELA (Edges='open' para estilo ABNT)
            # bbox: [esquerda, baixo, largura, altura]
            table_width = 0.85
            left_margin = 0.07
            the_table = ax.table(cellText=data, 
                                 loc='center', 
                                 cellLoc='center',
                                 edges='open',
                                 bbox=[left_margin, 0.15, table_width, 0.55]) 

            the_table.auto_set_font_size(False)
            the_table.set_fontsize(10)

            # 4. POSICIONAMENTO DOS CABEÇALHOS AGRUPADOS (Cálculo de centro)
            # A tabela tem 7 colunas (0 a 6). 
            # Papanicolaou cobre colunas 1, 2 e 3. Centro é a coluna 2.
            # Mamografia cobre colunas 4, 5 e 6. Centro é a coluna 5.
            
            y_group = 0.77  # Nível dos nomes dos exames
            y_line = 0.74   # Nível da linha separadora curta
            
            # Papanicolaou: Centralizado sobre as colunas de índice 1, 2 e 3
            plt.text(0.49, y_group, "Papanicolaou", ha='center', fontweight='bold', fontsize=11, transform=ax.transAxes)
            # Mamografia: Centralizado sobre as colunas de índice 4, 5 e 6
            plt.text(0.79, y_group, "Mamografia", ha='center', fontweight='bold', fontsize=11, transform=ax.transAxes)

            # 5. LINHAS HORIZONTAIS (Estilo Revista)
            # Linha superior (grossa)
            ax.plot([left_margin, left_margin + table_width], [0.82, 0.82], color='black', lw=1.5, transform=ax.transAxes) 
            
            # Linhas curtas sob os nomes (Simetricamente centralizadas)
            ax.plot([0.37, 0.61], [y_line, y_line], color='black', lw=1, transform=ax.transAxes) # Sob Pap
            ax.plot([0.67, 0.91], [y_line, y_line], color='black', lw=1, transform=ax.transAxes) # Sob Mam

            # Linha que separa cabeçalho dos dados
            ax.plot([left_margin, left_margin + table_width], [0.70, 0.70], color='black', lw=1, transform=ax.transAxes)
            
            # Linha de fechamento (base da página)
            ax.plot([left_margin, left_margin + table_width], [0.15, 0.15], color='black', lw=1.5, transform=ax.transAxes)

            # 6. AJUSTES FINOS DE ALINHAMENTO E ESPAÇAMENTO
            for i in range(len(data)):
                # COLUNA DECIL: Força alinhamento à esquerda e adiciona um pequeno recuo (padding)
                cell_decil = the_table[(i, 0)]
                cell_decil.set_text_props(ha='left')
                
                # Deixa a primeira linha (sub-cabeçalho) em negrito
                if i == 0:
                    for j in range(7):
                        the_table[(i, j)].set_text_props(fontweight='bold')
                
                # Dá altura para as células não ficarem espremidas
                for j in range(7):
                    the_table[(i, j)].set_height(0.06)

            # 7. FONTE (Rodapé)
            plt.text(left_margin, 0.11, f"Fonte: Elaboração própria a partir dos dados da PNS/{ano}, IBGE.", 
                     ha='left', fontsize=9, transform=ax.transAxes)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
    print(f"Sucesso! PDF gerado perfeitamente em: {filename}")

# Chamar a função
exportar_tabela5_final_perfeita(tabelas)

Sucesso! PDF gerado perfeitamente em: Tabela_05_Revista_Final.pdf
